In [1]:
"""
Decision Tree: Buy/Sell/Hold signal prediction
- Features : SMA(10), SMA(20), EMA(10), EMA(20), price/volume ratios
- Target   : 5-day forward return → Buy / Hold / Sell
- Training : incremental ticker-by-ticker (memory efficient)
"""

import sqlite3
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder


In [2]:

# ── Config ────────────────────────────────────────────────────────────────────
DB_PATH       = "dataset/stock_prices_20y.db"
MODEL_OUT     = "decision_tree_model.pkl"
HORIZON       = 5          # days forward for signal
BUY_THRESH    = 0.02       # +2 % → Buy
SELL_THRESH   = -0.02      # -2 % → Sell
TEST_RATIO    = 0.2        # last 20 % of each ticker used for eval
RANDOM_STATE  = 42


In [3]:

# ── Label generation ─────────────────────────────────────────────────────────
def make_signal(series: pd.Series, horizon=HORIZON) -> pd.Series:
    fwd_return = series.shift(-horizon) / series - 1
    conditions = [fwd_return >= BUY_THRESH, fwd_return <= SELL_THRESH]
    return pd.Series(
        np.select(conditions, ["Buy", "Sell"], default="Hold"),
        index=series.index,
    )


In [4]:

# ── Feature engineering ───────────────────────────────────────────────────────
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    c = df["close"]
    v = df["volume"]

    df["sma_10"]       = c.rolling(10).mean()
    df["sma_20"]       = c.rolling(20).mean()
    df["ema_10"]       = c.ewm(span=10, adjust=False).mean()
    df["ema_20"]       = c.ewm(span=20, adjust=False).mean()
    df["sma_ratio"]    = df["sma_10"] / df["sma_20"]      # golden/death cross proxy
    df["ema_ratio"]    = df["ema_10"] / df["ema_20"]
    df["price_sma20"]  = c / df["sma_20"]                 # price relative to trend
    df["price_ema20"]  = c / df["ema_20"]
    df["vol_sma10"]    = v / v.rolling(10).mean()         # volume relative strength

    df["signal"] = make_signal(c)
    return df.dropna()


In [5]:

FEATURE_COLS = [
    "sma_ratio", "ema_ratio",
    "price_sma20", "price_ema20",
    "vol_sma10",
]


In [6]:

# ── Load tickers ──────────────────────────────────────────────────────────────
def get_tickers(conn):
    return [r[0] for r in conn.execute("SELECT DISTINCT ticker FROM prices ORDER BY ticker")]


In [7]:

def load_ticker(conn, ticker):
    df = pd.read_sql_query(
        "SELECT date, open, high, low, close, adj_close, volume "
        "FROM prices WHERE ticker = ? ORDER BY date",
        conn, params=(ticker,), parse_dates=["date"],
    )
    df = df.set_index("date")
    return df


In [8]:

# ── Incremental training ───────────────────────────────────────────────────────
def train():
    conn = sqlite3.connect(DB_PATH)
    tickers = get_tickers(conn)
    print(f"Found {len(tickers)} tickers.\n")

    le = LabelEncoder().fit(["Buy", "Hold", "Sell"])
    model = DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    all_X_train, all_y_train = [], []
    all_X_test,  all_y_test  = [], []

    for i, ticker in enumerate(tickers, 1):
        try:
            df = load_ticker(conn, ticker)
            if len(df) < 60:          # need enough rows for indicators + horizon
                continue
            df = make_features(df)

            split = int(len(df) * (1 - TEST_RATIO))
            train_df = df.iloc[:split]
            test_df  = df.iloc[split:]

            all_X_train.append(train_df[FEATURE_COLS].values)
            all_y_train.append(le.transform(train_df["signal"]))
            all_X_test.append(test_df[FEATURE_COLS].values)
            all_y_test.append(le.transform(test_df["signal"]))

            if i % 20 == 0 or i == len(tickers):
                print(f"  Loaded {i}/{len(tickers)} tickers …")
        except Exception as e:
            print(f"  [WARN] Skipping {ticker}: {e}")

    conn.close()

    X_train = np.vstack(all_X_train)
    y_train = np.concatenate(all_y_train)
    X_test  = np.vstack(all_X_test)
    y_test  = np.concatenate(all_y_test)

    print(f"\nTraining on {len(X_train):,} samples …")
    model.fit(X_train, y_train)
    print("Training complete.\n")

    # ── Evaluation ────────────────────────────────────────────────────────────
    y_pred = model.predict(X_test)
    print("── Test-set Classification Report ──────────────────────────")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    print("── Confusion Matrix (rows=actual, cols=predicted) ───────────")
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
    print(cm_df, "\n")

    print("── Feature Importances ──────────────────────────────────────")
    for feat, imp in sorted(zip(FEATURE_COLS, model.feature_importances_),
                             key=lambda x: -x[1]):
        print(f"  {feat:<15} {imp:.4f}")

    # ── Save ──────────────────────────────────────────────────────────────────
    with open(MODEL_OUT, "wb") as f:
        pickle.dump({"model": model, "label_encoder": le, "features": FEATURE_COLS}, f)
    print(f"\nModel saved → {MODEL_OUT}")


In [9]:
train()

Found 1000 tickers.

  Loaded 20/1000 tickers …
  Loaded 40/1000 tickers …
  Loaded 60/1000 tickers …
  Loaded 80/1000 tickers …
  Loaded 100/1000 tickers …
  Loaded 120/1000 tickers …
  Loaded 140/1000 tickers …
  Loaded 160/1000 tickers …
  Loaded 180/1000 tickers …
  Loaded 200/1000 tickers …
  Loaded 220/1000 tickers …
  Loaded 240/1000 tickers …
  Loaded 260/1000 tickers …
  Loaded 280/1000 tickers …
  Loaded 300/1000 tickers …
  Loaded 320/1000 tickers …
  Loaded 340/1000 tickers …
  Loaded 360/1000 tickers …
  Loaded 380/1000 tickers …
  Loaded 400/1000 tickers …
  Loaded 420/1000 tickers …
  Loaded 440/1000 tickers …
  Loaded 460/1000 tickers …
  Loaded 480/1000 tickers …
  Loaded 500/1000 tickers …
  Loaded 520/1000 tickers …
  Loaded 540/1000 tickers …
  Loaded 560/1000 tickers …
  Loaded 580/1000 tickers …
  Loaded 600/1000 tickers …
  Loaded 620/1000 tickers …
  Loaded 640/1000 tickers …
  Loaded 660/1000 tickers …
  Loaded 680/1000 tickers …
  Loaded 700/1000 tickers …
  L